# Full DRIVE SGL experiment

This notebook runs the submission configuration: eight study groups, out-of-fold pseudo-labels, final SGL training, an identically configured manual-label baseline, eight-way self-ensemble evaluation, and signed metric deltas. Checkpoints and results are stored in Google Drive so the run can resume after a disconnect.

Before running: select **Runtime → Change runtime type → T4 GPU** (or a faster GPU).

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected. Select Runtime > Change runtime type > T4 GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT = Path('/content/Retinal-vessel-segmentation-sgl')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/A-cpu-rg/Retinal-vessel-segmentation-sgl.git', str(PROJECT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project ready at', PROJECT)

In [ ]:
DATA = PROJECT / 'data' / 'DRIVE'
MIRROR = Path('/content/Retina-Unet-DRIVE')
if not (DATA / 'test' / '1st_manual' / '01_manual1.gif').exists():
    if not MIRROR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/wfdubowen/Retina-Unet.git', str(MIRROR)], check=True)
    DATA.mkdir(parents=True, exist_ok=True)
    subprocess.run(['cp', '-R', str(MIRROR / 'DRIVE') + '/.', str(DATA)], check=True)

sys.path.insert(0, str(PROJECT / 'src'))
from dataset import DriveDataset
train_data = DriveDataset(DATA, split='training', mode='full')
test_data = DriveDataset(DATA, split='test', mode='full')
assert len(train_data) == len(test_data) == 20
print('DRIVE validated: 20 training and 20 test images with labels and masks')
del train_data, test_data

## Full training and evaluation

This is the long-running cell. It is safe to rerun: `--resume` continues compatible fold, SGL, and baseline checkpoints stored in Google Drive. Do not change the configuration between resumed runs.

In [ ]:
RUN = Path('/content/drive/MyDrive/retinal_sgl_full_run')
RUN.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, 'src/run_sgl.py',
    '--root', str(DATA),
    '--device', 'cuda',
    '--workers', '2',
    '--k', '8',
    '--fold-epochs', '15',
    '--final-epochs', '25',
    '--patch-size', '256',
    '--batch-size', '4',
    '--repeat', '40',
    '--with-baseline',
    '--resume',
    '--ckpt-dir', str(RUN / 'checkpoints'),
    '--teacher-dir', str(RUN / 'pseudo_labels'),
    '--save-dir', str(RUN / 'test_predictions'),
    '--results', str(RUN / 'sgl_test_results.json'),
    '--baseline-results', str(RUN / 'baseline_test_results.json'),
    '--comparison-results', str(RUN / 'comparison.json'),
]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=PROJECT, check=True)

In [ ]:
import json
import pandas as pd

comparison = json.loads((RUN / 'comparison.json').read_text())
metrics = ['acc', 'se', 'sp', 'auc', 'dice', 'iou_vessel']
labels = ['Accuracy', 'Sensitivity', 'Specificity', 'AUC', 'Dice / F1', 'Vessel IoU']
table = pd.DataFrame({
    'Metric': labels,
    'Baseline': [comparison['baseline']['fov'][m] for m in metrics],
    'SGL': [comparison['sgl']['fov'][m] for m in metrics],
    'SGL - baseline': [comparison['delta_sgl_minus_baseline']['fov'][m] for m in metrics],
})
display(table.style.format({c: '{:.4f}' for c in table.columns if c != 'Metric'}))
print('Results saved permanently in:', RUN)

In [ ]:
print('| Metric | Baseline | SGL | SGL - baseline |')
print('|---|---:|---:|---:|')
for _, row in table.iterrows():
    print(f"| {row['Metric']} | {row['Baseline']:.4f} | {row['SGL']:.4f} | {row['SGL - baseline']:+.4f} |")